In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 데이터 전처리

In [ ]:
import os
import pandas as pd
import pickle
import openpyxl

file_path = 'drive/MyDrive/데이터사이언스/goal_set.p'
goal_set = pickle.load(open(file_path, 'rb'))

train_data = goal_set["train"]
test_data = goal_set["test"]

df_train = pd.DataFrame(train_data)
df_test = pd.DataFrame(test_data)

df_train["explicit_inform_slots"] = df_train["goal"].apply(lambda x: f"'{next(iter(x.get('explicit_inform_slots', {}).keys()), '')}'")
df_train["implicit_inform_slots"] = df_train["goal"].apply(lambda x: [f"'{key}'" for key in x.get('implicit_inform_slots', {}).keys()])
df_train['combined_slots'] = df_train.apply(lambda row: [row['explicit_inform_slots']] + row['implicit_inform_slots'], axis=1)

df_test["explicit_inform_slots"] = df_test["goal"].apply(lambda x: f"'{next(iter(x.get('explicit_inform_slots', {}).keys()), '')}'")
df_test["implicit_inform_slots"] = df_test["goal"].apply(lambda x: [f"'{key}'" for key in x.get('implicit_inform_slots', {}).keys()])
df_test['combined_slots'] = df_test.apply(lambda row: [row['explicit_inform_slots']] + row['implicit_inform_slots'], axis=1)

# 작은따옴표를 제거하는 함수 정의
def remove_quotes(lst):
    return [item.replace("'", "") for item in lst]

df_train['combined_slots'] = df_train['combined_slots'].apply(remove_quotes)
df_test['combined_slots'] = df_test['combined_slots'].apply(remove_quotes)

# 필요 없는 열 제거
df_train.drop(columns=['explicit_inform_slots', 'implicit_inform_slots', 'goal', 'group_id'], inplace=True)
df_test.drop(columns=['explicit_inform_slots', 'implicit_inform_slots', 'goal', 'group_id'], inplace=True)

df_test.drop(columns='consult_id', inplace=True)
df_train.drop(columns='consult_id', inplace=True)

# combined_slots에서 모든 값을 모아서 유일한 값들을 출력합니다.
unique_train = set()
for slots_list in df_train['combined_slots']:
    unique_train.update(slots_list)

unique_train = list(unique_train)

# combined_slots에서 모든 값을 모아서 유일한 값들을 출력합니다.
unique_test = set()
for slots_list in df_test['combined_slots']:
    unique_test.update(slots_list)

unique_test = list(unique_test)

# 중복되지 않는 값을 찾습니다.
unique_train_not_in_test = [item for item in unique_train if item not in unique_test]

# 대상 문자열
target_strings_train = [
    'Skin dryness, peeling, scaliness, or roughness',
    'Muscle cramps, contractures, or spasms'
]

# 'combined_slots' 열의 각 리스트 값에 대해 반복하면서 쉼표 없애기
for i, slots_list in enumerate(df_train['combined_slots']):
    updated_slots_list = []
    for item in slots_list:
        # 대상 문자열이 있는 경우 쉼표 없애고 업데이트
        if item in target_strings_train:
            updated_slots_list.append(item.replace(',', ''))
        else:
            updated_slots_list.append(item)
    # 업데이트된 리스트로 교체
    df_train.at[i, 'combined_slots'] = updated_slots_list

target_strings_test = [
    'Skin dryness, peeling, scaliness, or roughness'
]

# 'combined_slots' 열의 각 리스트 값에 대해 반복하면서 쉼표 없애기
for i, slots_list in enumerate(df_test['combined_slots']):
    updated_slots_list = []
    for item in slots_list:
        # 대상 문자열이 있는 경우 쉼표 없애고 업데이트
        if item in target_strings_train:
            updated_slots_list.append(item.replace(',', ''))
        else:
            updated_slots_list.append(item)
    # 업데이트된 리스트로 교체
    df_test.at[i, 'combined_slots'] = updated_slots_list

train = df_train.copy()
test  = df_test.copy()

train.to_excel('train.xlsx', index=False)
test.to_excel('test.xlsx', index=False)

In [ ]:
df_train

,disease_tag,combined_slots
0,Central retinal artery or vein occlusion,"[Spots or clouds in vision, Diminished vision,..."
1,Degenerative disc disease,"[Shoulder pain, Back pain, Low back pain, Neck..."
2,Diabetic retinopathy,[Foreign body sensation in eye]
3,Chronic back pain,"[Low back pain, Back pain, Side pain]"
4,Air embolism,"[Wrist pain, Pain in eye, Shoulder cramps or s..."
...,...,...
23995,Concussion,"[Neck pain, Headache, Nausea]"
23996,Cushing syndrome,[Weight gain]
23997,Fibromyalgia,"[Low back pain, Back pain, Ache all over, Shou..."
23998,Dengue fever,"[Fever, Sore throat, Wrist pain, Pain during p..."


# TF-IDF 벡터화

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# 쉼표를 기준으로 문서를 토큰화하는 사용자 정의 토크나이저 함수
def custom_tokenizer(text):
    # 쉼표를 기준으로 분할한 뒤 앞뒤 공백을 제거하여 반환
    return [word.strip() for word in text.split(',')]

# TF-IDF 벡터화를 위한 객체 생성 (쉼표를 기준으로 토큰화하는 사용자 정의 토크나이저 지정)
tfidf_vectorizer = TfidfVectorizer(tokenizer=custom_tokenizer)

# 'combined_slots' 열의 값을 하나의 텍스트 문서로 합치기
documents_train = df_train['combined_slots'].apply(lambda x: ', '.join(x))

# TF-IDF 벡터화를 수행하고 결과를 반환
tfidf_matrix_train = tfidf_vectorizer.fit_transform(documents_train)

# TF-IDF 벡터화된 결과를 데이터프레임으로 변환하여 출력
df_tfidf_train = pd.DataFrame(tfidf_matrix_train.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
df_tfidf_train.set_index(df_train['disease_tag'], inplace=True)

# 'combined_slots' 열의 값을 하나의 텍스트 문서로 합치기
documents_test = df_test['combined_slots'].apply(lambda x: ', '.join(x))

# TF-IDF 벡터화를 수행하고 결과를 반환
tfidf_matrix_test = tfidf_vectorizer.transform(documents_test)

# TF-IDF 벡터화된 결과를 데이터프레임으로 변환하여 출력
df_tfidf_test = pd.DataFrame(tfidf_matrix_test.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
df_tfidf_test.set_index(df_test['disease_tag'], inplace=True)

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:528: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
# 각 질병 별로 GROUP BY하고 tf-idf 값을 평균화
df_tfidf_train_grouped = df_tfidf_train.groupby('disease_tag').mean()
df_tfidf_test_grouped = df_tfidf_test.groupby('disease_tag').mean()

In [ ]:
df_tfidf_train_grouped

,abnormal appearing skin,abnormal involuntary movements,abnormal movement of eyelid,abnormal size or shape of ear,absence of menstruation,abusing alcohol,ache all over,acne or pimples,allergic reaction,ankle pain,...,warts,weakness,weight gain,white discharge from eye,wrinkles on skin,wrist lump or mass,wrist pain,wrist stiffness or tightness,wrist swelling,wrist weakness
disease_tag,,,,,,,,,,,,,,,,,,,,,
Acanthosis nigricans,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.111624,0.076855,0.000000,...,0.075482,0.000000,0.559358,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000
Acariasis,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.017788
Acne,0.093737,0.000000,0.0,0.0,0.0,0.0,0.000000,0.679500,0.000000,0.000000,...,0.060450,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000
Actinic keratosis,0.375126,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.041507,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000
Acute glaucoma,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Gonorrhea,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000
Gout,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.162145,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.123107,0.0,0.018926,0.000000
Granuloma inguinale,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.055256


In [ ]:
df_train_transposed = df_tfidf_train_grouped.transpose()
df_test_transposed = df_tfidf_test_grouped.transpose()

In [ ]:
df_train_transposed

disease_tag,Acanthosis nigricans,Acariasis,Acne,Actinic keratosis,Acute glaucoma,Acute kidney injury,Acute stress reaction,Adhesive capsulitis of the shoulder,Adjustment reaction,Air embolism,...,Fluid overload,Fracture of the pelvis,Fracture of the rib,Ganglion cyst,Gas gangrene,Gonorrhea,Gout,Granuloma inguinale,Graves disease,Guillain Barre syndrome
abnormal appearing skin,0.0,0.000000,0.093737,0.375126,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
abnormal involuntary movements,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.069787,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.072555,0.0
abnormal movement of eyelid,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
abnormal size or shape of ear,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
absence of menstruation,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wrist lump or mass,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.187241,0.000000,0.0,0.000000,0.000000,0.000000,0.0
wrist pain,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.257685,...,0.0,0.0,0.0,0.225993,0.136913,0.0,0.123107,0.000000,0.000000,0.0
wrist stiffness or tightness,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
wrist swelling,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.063342,0.000000,0.0,0.018926,0.000000,0.000000,0.0


# 질병 예측 모델

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 학습 데이터와 타겟 데이터 분리
X_train = df_tfidf_train
y_train = df_tfidf_train.index

X_test = df_tfidf_test
y_test = df_tfidf_test.index


# 로지스틱 회귀 모델 학습
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 검증 데이터에 대한 예측
y_pred = model.predict(X_test)

# 모델 평가
print(classification_report(y_test, y_pred))

                                          precision    recall  f1-score   support

                    Acanthosis nigricans       0.77      0.90      0.83        61
                               Acariasis       0.73      0.92      0.81        72
                                    Acne       0.84      0.96      0.90        73
                       Actinic keratosis       0.69      0.80      0.74        79
                          Acute glaucoma       0.30      0.36      0.33        58
                     Acute kidney injury       0.86      0.80      0.83        75
                   Acute stress reaction       0.92      0.69      0.79        70
     Adhesive capsulitis of the shoulder       0.87      0.98      0.92        54
                     Adjustment reaction       0.71      0.65      0.68        83
                            Air embolism       0.45      0.07      0.13        67
                    Alcohol intoxication       0.88      0.88      0.88        68
               

In [ ]:
# 입력 증상 TF-IDF 벡터화 함수
def transform_input_symptoms(input_symptoms, tfidf_vectorizer):
    input_text = ', '.join(input_symptoms)
    input_vector = tfidf_vectorizer.transform([input_text])
    return input_vector

# 예측 함수
def predict_top_diseases(input_symptoms, tfidf_vectorizer, model, top_n=5):
    input_vector = transform_input_symptoms(input_symptoms, tfidf_vectorizer)
    probabilities = model.predict_proba(input_vector)[0]
    top_indices = probabilities.argsort()[-top_n:][::-1]
    top_diseases = [model.classes_[i] for i in top_indices]
    top_probabilities = [probabilities[i] for i in top_indices]
    return list(zip(top_diseases, top_probabilities))

# 입력된 증상으로부터 상위 N개의 질병 예측
input_symptoms = ['white discharge from eye','pain in eye','eye redness','foreign body sensation in eye']
top_diseases = predict_top_diseases(input_symptoms, tfidf_vectorizer, model, top_n=5)
print(f"Top {len(top_diseases)} predicted diseases for input '{input_symptoms}':")
print(top_diseases)
for disease, probability in top_diseases:
    print(f"{disease}: {probability:.4f}")

Top 5 predicted diseases for input '['white discharge from eye', 'pain in eye', 'eye redness', 'foreign body sensation in eye']':
[('Corneal disorder', 0.37618681153109157), ('Corneal abrasion', 0.2969729530210432), ('Ectropion', 0.21925453630513272), ('Acute glaucoma', 0.022448971032823214), ('Chalazion', 0.012354576279055939)]
Corneal disorder: 0.3762
Corneal abrasion: 0.2970
Ectropion: 0.2193
Acute glaucoma: 0.0224
Chalazion: 0.0124


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [ ]:
import math
import numpy as np

def dcg(rel,i) :
    return rel/(math.log2(i+1))

def idcg(rel,t) :
    if t==0 :
        return 0
    else :
        return dcg(rel,t)+idcg(rel,t-1)

def ndcg_n_cal (p,label,n) :
    if n==0 :
        return 0
    elif n<=len(p) and p[n-1] in label :
        return dcg(1,n)+ndcg_n_cal(p,label,n-1)
    else:
        return ndcg_n_cal(p,label,n-1)

def ndcg_n(p,label,n) :
    return ndcg_n_cal(p,label,n) / idcg(1,len(label))

def do_ndcg(input,label,n) :
  top_diseases = predict_top_diseases(input, tfidf_vectorizer, model, top_n=5)
  extracted_top = [item[0] for item in top_diseases]
  ndcg_score=ndcg_n(extracted_top,[label],n)
  return ndcg_score

In [ ]:
#input_symptoms = ['abnormal appearing skin', 'anxiety and nervousness','fever']
#top_diseases = predict_top_diseases(input_symptoms, tfidf_vectorizer, model, top_n=5)
#extracted_top = [item[0] for item in top_diseases]
#answer=['Amyloidosis']
#n=5
#ndcg_n(extracted_top,answer,n)

data = pd.read_csv('/content/drive/MyDrive/Task2.csv')
results = []
n=5
for index, row in data.iterrows():
    answer_disease = row['disease']
    input_disease = eval(row['inform_slots'])
    ndcg_score = do_ndcg(input_disease, answer_disease, n)
    results.append(ndcg_score)

print(results)
print("Task2의 최종 평가 데이터 NDCG@5 결과 : ",np.mean(results))

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.6309297535714575, 0.6309297535714575, 0.5, 0.43067655807339306, 0.5, 0.5, 0.43067655807339306, 0.43067655807339306, 0.6309297535714575, 1.0, 0.6309297535714575, 1.0, 0.5, 1.0, 0.5, 0.43067655807339306, 0.43067655807339306, 0.5, 0.6309297535714575, 1.0, 0.6309297535714575, 0.43067655807339306, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.6309297535714575, 1.0, 1.0, 1.0, 0.430676